# Low-level nonlinear registration

Use this notebook when the moving image is already affine-aligned to the fixed image. The nonlinear kernel returns a registered scalar image and one unique disposable warp candidate; this notebook promotes that candidate atomically and always releases it.

In [ ]:
# EDIT THESE PATHS before running the notebook.
from pathlib import Path

FIXED_IMAGE = Path("path/to/fixed_reference.tif")
AFFINE_ALIGNED_MOVING_IMAGE = Path("path/to/affine_aligned_moving.tif")
WARP_OUTPUT = Path("path/to/nonlinear_warp_2.nii.gz")
REGISTERED_OUTPUT = Path("path/to/nonlinear_registered_2.tif")
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
ITK_THREADS = 1
BLAS_THREADS = 1

In [ ]:
import importlib
import shutil
from contextlib import suppress

from cyexm.io.atomic import atomic_write
from cyexm.io.images import load_ants_image, zero_mask
from cyexm.io.tiff import write_uint16_tiff
from cyexm.registration.nonlinear import register_image
from cyexm.runtime import NativeThreadConfig, configure_native_threads

NATIVE_THREADS = NativeThreadConfig(ITK_THREADS, BLAS_THREADS)
configure_native_threads(NATIVE_THREADS)
ants = importlib.import_module("ants")
np = importlib.import_module("numpy")

fixed = load_ants_image(FIXED_IMAGE, PIXEL_SIZE_XYZ)
moving = load_ants_image(AFFINE_ALIGNED_MOVING_IMAGE, PIXEL_SIZE_XYZ)

In [ ]:
registered, candidate_warp = register_image(
    moving_image=moving,
    fixed_image=fixed,
    moving_mask=zero_mask(moving),
    fixed_mask=zero_mask(fixed),
    accuracy="high",
    verbose=True,
    native_threads=NATIVE_THREADS,
)
try:
    atomic_write(
        WARP_OUTPUT,
        lambda path: shutil.copyfile(candidate_warp, path),
        lambda path: ants.image_read(str(path)).dimension == 3,
    )
finally:
    with suppress(OSError):
        candidate_warp.unlink(missing_ok=True)
values = registered.numpy()
write_uint16_tiff(
    registered,
    REGISTERED_OUTPUT,
    float(values.min()),
    float(values.max()),
)

A displacement field has three components in its final array axis. Magnitudes below are in the same physical units as the image spacing (micrometers here).

In [ ]:
warp_values = ants.image_read(str(WARP_OUTPUT)).numpy()
magnitude = np.linalg.norm(warp_values, axis=-1)
{
    "mean_displacement_um": float(magnitude.mean()),
    "maximum_displacement_um": float(magnitude.max()),
}